# LeetCode 2236 — Root Equals Sum of Children bằng Python

Link đề: https://leetcode.com/problems/root-equals-sum-of-children/ · Độ khó: **Easy** · Chủ đề: Tree, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node (LeetCode cung cấp sẵn):
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def checkTree(self, root: Optional[TreeNode]) -> bool:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu **cây nhị phân** là gì: mỗi nút có giá trị `val`, con trái `left` và con phải `right` (có thể là `None`).
2. Tự định nghĩa `class TreeNode` và viết hàm dựng cây từ danh sách theo thứ tự tầng (level-order) để **chạy thử trong notebook**.
3. Truy cập con của nút bằng `root.left.val`, `root.right.val` và biết khi nào nút con có thể là `None`.
4. Viết 4–5 cách: so sánh trực tiếp, hàm giúp `val(node)` an toàn với `None`, duyệt BFS/DFS thu thập giá trị, và kiểm tra đệ quy tổng quát.
5. Trace, test đối chiếu mọi cách, và tránh lỗi: quên `.val`, `None.val`, so sánh nút với số.

## 2. Đề bài & ví dụ

Cho `root` của một cây nhị phân gồm **đúng 3 nút**: nút gốc, nút con trái và nút con phải. Trả về `True` nếu **giá trị của gốc bằng tổng giá trị hai con**, ngược lại `False`.

| Ví dụ | root (level-order) | Output | Giải thích |
|---|---|---|---|
| 1 | `[10,4,6]` | `True` | `10 == 4 + 6` |
| 2 | `[5,3,1]` | `False` | `5 != 3 + 1` |

Hình dạng cây ví dụ 1:

```
    10
   /  \
  4    6
```

Constraints: cây có đúng `3` nút; `-100 <= Node.val <= 100`.

Đề đảm bảo cây gồm gốc cùng con trái và con phải, nên `root.left` và `root.right` **luôn tồn tại** (không phải `None`).

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

root = TreeNode(10, TreeNode(4), TreeNode(6))
print(root.val, root.left.val, root.right.val)
print("10 == 4 + 6 ?", root.val == root.left.val + root.right.val)

10 4 6
10 == 4 + 6 ? True


## 3. Kiến thức nền

### 3.1. Lớp (class) và đối tượng

`class TreeNode` là "khuôn". `TreeNode(10, left, right)` tạo một **đối tượng** (nút) có ba thuộc tính: `val`, `left`, `right`. Truy cập thuộc tính bằng dấu chấm: `node.val`. `__init__` là hàm khởi tạo, `self` là chính đối tượng đang tạo. Tham số có **giá trị mặc định** (`val=0`, `left=None`) nên có thể bỏ qua khi không cần.

In [2]:
leaf = TreeNode(7)
print(leaf.val, leaf.left, leaf.right)
node = TreeNode(1, TreeNode(2), None)
print(node.left.val, node.right)

7 None None
2 None


### 3.2. `None` — "không có gì"

`None` là giá trị đặc biệt biểu thị **nút con không tồn tại**. Truy cập `None.val` gây lỗi `AttributeError`. Muốn an toàn, kiểm tra `if node is not None` (hoặc `if node:`) trước khi lấy `.val`.

In [3]:
node = TreeNode(1, TreeNode(2), None)
try:
    print(node.right.val)
except AttributeError as e:
    print("lỗi:", e)
print(node.right is None, node.left is None)

lỗi: 'NoneType' object has no attribute 'val'
True False


### 3.3. Dựng cây từ danh sách level-order

LeetCode mô tả cây bằng danh sách theo **từng tầng**, từ trái sang phải, `None` cho nút trống. `[10,4,6]` nghĩa là gốc `10`, con trái `4`, con phải `6`. Ta viết `build_tree(vals)` dùng **hàng đợi** (`collections.deque`): lấy một nút ra khỏi hàng, gán hai giá trị kế tiếp của danh sách làm con trái/phải, rồi đưa hai nút con (nếu có) vào hàng đợi.

In [4]:
from collections import deque

def build_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root

t = build_tree([10, 4, 6])
print(t.val, t.left.val, t.right.val)
print(build_tree([]) is None)

10 4 6
True


### 3.4. `if` / `else` và trả về `bool`

Phép so sánh `a == b` đã cho ra `True`/`False`, nên **không cần** viết `if a == b: return True else: return False`; chỉ cần `return a == b`. Tuy nhiên hãy hiểu cả hai cách viết tương đương.

In [5]:
a, b = 10, 4 + 6
def verbose(a, b):
    if a == b:
        return True
    else:
        return False
def short(a, b):
    return a == b
print(verbose(a, b), short(a, b), verbose(5, 4), short(5, 4))

True True False False


### 3.5. Duyệt cây bằng đệ quy và bằng hàng đợi

Cây là cấu trúc **đệ quy**: mỗi nút con lại là gốc của một cây con. Có thể duyệt bằng hàm gọi chính nó (`dfs(node.left)`) hoặc bằng hàng đợi (BFS) từng tầng. Với cây 3 nút chỉ có hai tầng.

In [6]:
def preorder(node):
    if node is None:
        return []
    return [node.val] + preorder(node.left) + preorder(node.right)

def level_order(root):
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        out.append(node.val)
        if node.left: q.append(node.left)
        if node.right: q.append(node.right)
    return out

t = build_tree([10, 4, 6])
print(preorder(t), level_order(t))

[10, 4, 6] [10, 4, 6]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Chỉ có ba số: `gốc`, `trái`, `phải`. Đề hỏi một đẳng thức: `gốc == trái + phải`?

Ví dụ 1: `10` so với `4 + 6 = 10` → bằng nhau → `True`.
Ví dụ 2: `5` so với `3 + 1 = 4` → khác → `False`.

Với số âm: `[0, -1, 1]` có `0 == -1 + 1` nên `True`; `[-5, -2, -3]` có `-5 == -2 + -3` nên `True`. Đừng giả sử mọi giá trị dương.

Vậy toàn bộ bài toán gồm: (1) lấy được ba giá trị từ cây, (2) cộng hai con và so sánh. Phần "cây" chỉ là cách truy cập: `root.val`, `root.left.val`, `root.right.val`.

In [7]:
for vals in ([10, 4, 6], [5, 3, 1], [0, -1, 1], [-5, -2, -3]):
    t = build_tree(vals)
    print(vals, "->", t.val, "vs", t.left.val, "+", t.right.val, "=", t.left.val + t.right.val, "|", t.val == t.left.val + t.right.val)

[10, 4, 6] -> 10 vs 4 + 6 = 10 | True
[5, 3, 1] -> 5 vs 3 + 1 = 4 | False
[0, -1, 1] -> 0 vs -1 + 1 = 0 | True
[-5, -2, -3] -> -5 vs -2 + -3 = -5 | True


## 5. Cách 1 — Brute Force / baseline: duyệt cây, thu thập ba giá trị rồi so sánh

Chưa dùng giả sử "đúng ba nút". Ta duyệt cây (preorder) để gom **mọi giá trị** vào một list rồi lấy ba phần tử đầu: `vals[0]` là gốc (preorder duyệt gốc trước), `vals[1]` là con trái, `vals[2]` là con phải.

In [8]:
def check_by_preorder(root):
    vals = []
    def dfs(node):
        if node is None:
            return
        vals.append(node.val)
        dfs(node.left)
        dfs(node.right)
    dfs(root)
    return vals[0] == vals[1] + vals[2]

print(check_by_preorder(build_tree([10, 4, 6])), check_by_preorder(build_tree([5, 3, 1])))

True False


In [9]:
from typing import List, Optional

class SolutionBrute:
    def checkTree(self, root: Optional[TreeNode]) -> bool:
        vals = []
        def dfs(node):
            if node is None:
                return
            vals.append(node.val)
            dfs(node.left)
            dfs(node.right)
        dfs(root)
        return vals[0] == vals[1] + vals[2]

print(SolutionBrute().checkTree(build_tree([10, 4, 6])))
print(SolutionBrute().checkTree(build_tree([5, 3, 1])))

True
False


**Độ phức tạp:** Gọi `n` là số nút. Với đề này `n = 3`. Tổng quát duyệt `n` nút: **Time `O(n)`**, **Space `O(n)`** (list và ngăn xếp đệ quy).

**Nút thắt:** ta duyệt cả cây rồi mới so sánh, trong khi đề đã đảm bảo chỉ có 3 nút và ta biết chính xác vị trí của chúng. Có thể truy cập thẳng.

## 6. Ý tưởng thuật toán / bước đột phá

Insight: vì cây luôn có **đúng ba nút** (gốc và hai con), cả ba giá trị truy cập trực tiếp qua `root`, không cần duyệt:

- `root.val` là giá trị gốc,
- `root.left.val` là con trái,
- `root.right.val` là con phải.

Quy tắc: **`root.val == root.left.val + root.right.val` thì `True`, ngược lại `False`.** Kết quả trong `O(1)` thời gian và `O(1)` bộ nhớ. Nếu muốn viết mã **chịu lỗi** (khi con có thể là `None`), dùng hàm giúp `value(node)` trả `0` cho `None`.

## 7. Các cách tiếp theo

### Cách 2 — So sánh trực tiếp (cách chuẩn)

Một dòng duy nhất, đúng với ràng buộc đề.

In [10]:
from typing import Optional

class SolutionDirect:
    def checkTree(self, root: Optional[TreeNode]) -> bool:
        return root.val == root.left.val + root.right.val

print(SolutionDirect().checkTree(build_tree([10, 4, 6])))
print(SolutionDirect().checkTree(build_tree([5, 3, 1])))

True
False


**Độ phức tạp:** Time `O(1)`, Space `O(1)`.

### Cách 3 — Hàm giúp `value(node)` an toàn với `None`

Định nghĩa `value(node) = node.val if node else 0`. Nhờ đó cùng đoạn code đúng cả khi một con bị thiếu (một nút con `None` được coi như `0`). Tổng quát hơn so với đề và rất hữu ích trong bài cây.

In [11]:
from typing import Optional

class SolutionSafe:
    def checkTree(self, root: Optional[TreeNode]) -> bool:
        def value(node):
            return node.val if node else 0
        return root.val == value(root.left) + value(root.right)

print(SolutionSafe().checkTree(build_tree([10, 4, 6])))
print(SolutionSafe().checkTree(build_tree([5, 3, 1])))
print("cây chỉ có con trái:", SolutionSafe().checkTree(build_tree([4, 4])))   # 4 == 4 + 0

True
False
cây chỉ có con trái: True


**Độ phức tạp:** Time `O(1)`, Space `O(1)`.

### Cách 4 — Duyệt BFS bằng hàng đợi (`deque`)

Duyệt theo tầng: phần tử đầu `vals[0]` là gốc, hai phần tử sau là hai con. Đây là cách "tổng quát" giống Cách 1 nhưng bằng vòng lặp, không đệ quy.

In [12]:
from collections import deque
from typing import Optional

class SolutionBFS:
    def checkTree(self, root: Optional[TreeNode]) -> bool:
        vals = []
        q = deque([root])
        while q:
            node = q.popleft()
            vals.append(node.val)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        return vals[0] == vals[1] + vals[2]

print(SolutionBFS().checkTree(build_tree([10, 4, 6])))
print(SolutionBFS().checkTree(build_tree([5, 3, 1])))

True
False


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (hàng đợi và list). Với `n = 3` là hằng số.

### Cách 5 (mở rộng) — Kiểm tra đệ quy "mọi nút trong bằng tổng hai con"

Bài toán tổng quát hóa: **mọi nút không phải lá** phải bằng tổng giá trị các con của nó (tương tự "Sum Tree"). Với cây 3 nút chỉ có gốc là nút trong, nên cho kết quả giống nhau. Đây là bước rèn luyện tư duy đệ quy: nút lá luôn đúng, nút trong đúng khi **tổng con khớp** và **cả hai cây con cũng thỏa**.

In [13]:
from typing import Optional

class SolutionRecursive:
    def checkTree(self, root: Optional[TreeNode]) -> bool:
        def ok(node):
            if node is None or (node.left is None and node.right is None):
                return True
            total = (node.left.val if node.left else 0) + (node.right.val if node.right else 0)
            return node.val == total and ok(node.left) and ok(node.right)
        return ok(root)

print(SolutionRecursive().checkTree(build_tree([10, 4, 6])))
print(SolutionRecursive().checkTree(build_tree([5, 3, 1])))
deep = build_tree([10, 4, 6, 1, 3])          # nút 4 có hai con 1 và 3 -> 4 == 1 + 3
print("cây sâu hơn (mở rộng):", SolutionRecursive().checkTree(deep))

True
False
cây sâu hơn (mở rộng): True


**Độ phức tạp:** Time `O(n)`, Space `O(h)` với `h` là chiều cao cây (ngăn xếp đệ quy). Với đề này `h = 2`.

## 8. TRACE từng bước

Trace Cách 2 với `[10, 4, 6]`:

| bước | biểu thức | giá trị |
|---|---|---|
| 1 | `root.val` | 10 |
| 2 | `root.left.val` | 4 |
| 3 | `root.right.val` | 6 |
| 4 | `4 + 6` | 10 |
| 5 | `10 == 10` | `True` |

Trace Cách 4 (BFS) cho `[5, 3, 1]`:

| bước | `q` trước | nút lấy ra | `vals` | `q` sau |
|---|---|---|---|---|
| 1 | `[5]` | 5 | `[5]` | `[3, 1]` |
| 2 | `[3, 1]` | 3 | `[5, 3]` | `[1]` |
| 3 | `[1]` | 1 | `[5, 3, 1]` | `[]` |

Kết luận `5 == 3 + 1`? `False`.

In [14]:
def trace_bfs(vals_in):
    root = build_tree(vals_in)
    q = deque([root])
    vals = []
    step = 0
    while q:
        step += 1
        before = [n.val for n in q]
        node = q.popleft()
        vals.append(node.val)
        if node.left: q.append(node.left)
        if node.right: q.append(node.right)
        print(f"bước {step}: q trước={before} -> lấy {node.val}; vals={vals}; q sau={[n.val for n in q]}")
    res = vals[0] == vals[1] + vals[2]
    print(f"{vals[0]} == {vals[1]} + {vals[2]} ? {res}")
    return res

trace_bfs([10, 4, 6])
trace_bfs([5, 3, 1])

bước 1: q trước=[10] -> lấy 10; vals=[10]; q sau=[4, 6]
bước 2: q trước=[4, 6] -> lấy 4; vals=[10, 4]; q sau=[6]
bước 3: q trước=[6] -> lấy 6; vals=[10, 4, 6]; q sau=[]
10 == 4 + 6 ? True
bước 1: q trước=[5] -> lấy 5; vals=[5]; q sau=[3, 1]
bước 2: q trước=[3, 1] -> lấy 3; vals=[5, 3]; q sau=[1]
bước 3: q trước=[1] -> lấy 1; vals=[5, 3, 1]; q sau=[]
5 == 3 + 1 ? False


False

In [15]:
def trace_direct(vals_in):
    root = build_tree(vals_in)
    print("root.val =", root.val)
    print("root.left.val =", root.left.val)
    print("root.right.val =", root.right.val)
    total = root.left.val + root.right.val
    print("tổng hai con =", total, "->", root.val == total)
    return root.val == total

trace_direct([10, 4, 6])
trace_direct([-5, -2, -3])

root.val = 10
root.left.val = 4
root.right.val = 6
tổng hai con = 10 -> True
root.val = -5
root.left.val = -2
root.right.val = -3
tổng hai con = -5 -> True


True

## 9. Code nộp LeetCode

Trên LeetCode, `TreeNode` được cung cấp sẵn nên bạn **không** cần định nghĩa lại; trong notebook ta đã định nghĩa ở trên.

In [16]:
from typing import Optional

class Solution:
    def checkTree(self, root: Optional[TreeNode]) -> bool:
        return root.val == root.left.val + root.right.val

print(Solution().checkTree(build_tree([10, 4, 6])))
print(Solution().checkTree(build_tree([5, 3, 1])))

True
False


## 10. Vì sao thuật toán đúng

- **Cấu trúc đảm bảo:** đề nêu rõ cây gồm **gốc, con trái và con phải** (đúng ba nút theo cấu trúc đó). Vì vậy `root.left` và `root.right` đều khác `None` và ba giá trị truy cập trực tiếp được.
- **Định nghĩa:** "giá trị gốc bằng tổng giá trị hai con" chính là biểu thức `root.val == root.left.val + root.right.val` — trả về `True` đúng lúc đề mong muốn.
- **Số âm và số 0:** phép cộng và so sánh số nguyên hoạt động bình thường với số âm, nên không cần xử lý riêng.
- **Cách mở rộng** (hàm `value` hoặc kiểm tra đệ quy) đúng vì với cây 3 nút chúng quy về cùng biểu thức; ở cây tổng quát chúng cho kết quả theo định nghĩa mở rộng ("mỗi nút trong bằng tổng con").

## 11. Phân tích Time / Space Complexity

Đặt `n` là số nút (với đề này `n = 3` cố định), `h` là chiều cao cây.

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt preorder đệ quy | `O(n)` | `O(n)` |
| 2. So sánh trực tiếp | `O(1)` | `O(1)` |
| 3. Hàm giúp `value(node)` | `O(1)` | `O(1)` |
| 4. BFS bằng `deque` | `O(n)` | `O(n)` |
| 5. Đệ quy "mọi nút trong" | `O(n)` | `O(h)` |

Vì `n = 3` nên mọi cách đều là hằng số. Cách 2 tối ưu và ngắn nhất.

## 12. Bộ test

Dùng `build_tree` để tạo cây từ danh sách level-order. Test gồm ví dụ đề, số âm, số 0, và test ngẫu nhiên đối chiếu **tất cả cách**.

In [17]:
import random

solvers = [SolutionBrute(), SolutionDirect(), SolutionSafe(), SolutionBFS(), SolutionRecursive(), Solution()]

fixed = [
    ([10, 4, 6], True),
    ([5, 3, 1], False),
    ([0, 0, 0], True),              # toàn số 0
    ([0, -1, 1], True),             # tổng bằng 0
    ([-5, -2, -3], True),           # toàn số âm
    ([-100, -50, -50], True),       # biên dưới
    ([100, 50, 50], True),          # biên trên
    ([100, 100, 100], False),       # phản ví dụ: gốc bằng từng con nhưng không bằng tổng
    ([1, 1, 0], True),              # một con bằng 0
]
for vals, expected in fixed:
    for sol in solvers:
        assert sol.checkTree(build_tree(vals)) == expected, (type(sol).__name__, vals)
print("test cố định: OK")

random.seed(2236)
for _ in range(500):
    a, b = random.randint(-100, 100), random.randint(-100, 100)
    if random.random() < 0.5:
        r = a + b
        r = max(-100, min(100, r))
    else:
        r = random.randint(-100, 100)
    vals = [r, a, b]
    expected = r == a + b
    for sol in solvers:
        assert sol.checkTree(build_tree(vals)) == expected, (type(sol).__name__, vals)
print("500 test ngẫu nhiên: OK")

test cố định: OK
500 test ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên `.val`

`root.left + root.right` cộng **hai đối tượng nút**, không phải hai số → `TypeError`.

In [18]:
root = build_tree([10, 4, 6])
try:
    print(root.val == root.left + root.right)
except TypeError as e:
    print("sai -> TypeError:", e)
print("đúng:", root.val == root.left.val + root.right.val)

sai -> TypeError: unsupported operand type(s) for +: 'TreeNode' and 'TreeNode'
đúng: True


### Lỗi 2: So sánh nút với số

`root == 10` so sánh **đối tượng** với số nguyên, luôn `False` (trừ khi định nghĩa `__eq__`).

In [19]:
root = build_tree([10, 4, 6])
print("sai :", root == 10, "| đúng:", root.val == 10)

sai : False | đúng: True


### Lỗi 3: Truy cập `.val` của `None`

Với dữ liệu thật đề đảm bảo đủ ba nút, nhưng khi tự làm biến thể (cây thiếu con) sẽ lỗi.

In [20]:
root = build_tree([4, 4])              # chỉ có con trái
try:
    print(root.val == root.left.val + root.right.val)
except AttributeError as e:
    print("sai -> AttributeError:", e)
print("đúng (hàm giúp):", SolutionSafe().checkTree(root))

sai -> AttributeError: 'NoneType' object has no attribute 'val'
đúng (hàm giúp): True


### Lỗi 4: Nhầm `=` và `==`

`=` là **gán**, `==` là **so sánh**. Viết `return root.val = ...` là lỗi cú pháp; viết `if x = y:` cũng vậy. Hãy dùng `==`.

In [21]:
x = 10
y = 4 + 6
print(x == y)
try:
    compile("print(x = y)", "<demo>", "eval")        # không lỗi cú pháp nhưng nghĩa khác (đối số tên)
    print("print(x = y) -> truyền đối số tên x, không phải so sánh")
except SyntaxError as e:
    print("lỗi cú pháp:", e)
try:
    compile("if x = y: pass", "<demo>", "exec")
except SyntaxError:
    print("`if x = y:` là SyntaxError; phải viết `if x == y:`")

True
print(x = y) -> truyền đối số tên x, không phải so sánh
`if x = y:` là SyntaxError; phải viết `if x == y:`


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đọc kỹ constraints: "cây có **đúng 3 nút**" cho phép truy cập trực tiếp `root.left.val`, `root.right.val`.
- Bài cây quen thuộc: truy cập con bằng `.left` / `.right`; kiểm tra `None` trước khi `.val` khi cây không bị ràng buộc.
- `return biểu_thức_bool` thay cho `if ... return True else return False`.
- Muốn debug cây: viết `build_tree` và `level_order` như trong notebook này để dựng và in cây.
- Bài tổng quát hơn dùng **đệ quy**: giải cho nút hiện tại + kết quả cây con trái + cây con phải.

## 15. Biến thể của bài

1. Cây tổng quát (không chỉ 3 nút): kiểm tra **mọi nút trong** bằng tổng con (Sum Tree).
2. LeetCode 112 Path Sum: có đường từ gốc đến lá có tổng bằng `targetSum`?
3. LeetCode 104 Maximum Depth of Binary Tree, 100 Same Tree, 101 Symmetric Tree.
4. Giá trị gốc bằng **tổng cả cây con** (không chỉ con trực tiếp).
5. Gốc bằng **tích** hai con, hoặc **gốc lớn hơn** cả hai con (kiểm tra tính chất max-heap).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Preorder đệ quy | gom giá trị rồi so | `O(n)` | `O(n)` | học duyệt cây, không biết trước kích thước |
| 2. So sánh trực tiếp | `root.val == l.val + r.val` | `O(1)` | `O(1)` | **khuyến nghị khi nộp** |
| 3. Hàm giúp `value` | `None` coi như `0` | `O(1)` | `O(1)` | biến thể cây thiếu con |
| 4. BFS `deque` | gom giá trị theo tầng | `O(n)` | `O(n)` | học BFS, cây tổng quát |
| 5. Đệ quy mọi nút trong | mỗi nút trong bằng tổng con | `O(n)` | `O(h)` | bài Sum Tree mở rộng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `build_tree([7, 3, 4])` rồi `print(root.val == root.left.val + root.right.val)` in ra gì?

**Bài 2 (tính tay).** Cho cây `[-3, -1, -2]` và `[0, 5, -5]`. Mỗi cây trả về gì?

**Bài 3 (điền TODO).** Hoàn thành hàm kiểm tra gốc có **lớn hơn cả hai con** không.

In [22]:
def root_greater_than_children(root):
    # TODO: root.val > root.left.val and root.val > root.right.val
    pass

# chưa gọi hàm ở đây để notebook chạy không lỗi

**Bài 4 (tự viết lại).** Viết `count_nodes(root)` đếm số nút của cây bằng đệ quy (không dùng giả sử 3 nút).

In [23]:
def count_nodes(root):
    # TODO: tự viết, nút None có 0 nút
    pass

**Bài 5 (mini challenge).** Viết `is_sum_tree(root)` trả về `True` nếu **mọi nút không phải lá** có giá trị bằng tổng giá trị hai con (con thiếu coi như `0`). Ví dụ `[10,4,6,1,3]` là `True` còn `[10,4,6,1,2]` là `False`.

In [24]:
def is_sum_tree(root):
    # TODO: lá luôn True; nút trong cần val == tổng con và cả hai cây con đều thỏa
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `3 + 4 = 7`. Bài 2: tính `-1 + -2` và `5 + -5`.
- **Mức 2 (vừa):** Bài 3 viết một biểu thức `and`. Bài 4: `1 + count_nodes(left) + count_nodes(right)`.
- **Mức 3 (sâu):** Bài 5: dùng hàm giúp `val(node) = node.val if node else 0`; điều kiện dừng: `node is None` hoặc là lá trả `True`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [25]:
# Bài 1 và 2
r1 = build_tree([7, 3, 4])
assert (r1.val == r1.left.val + r1.right.val) is True
assert Solution().checkTree(build_tree([-3, -1, -2])) is True
assert Solution().checkTree(build_tree([0, 5, -5])) is True

# Bài 3
def root_greater_than_children(root):
    return root.val > root.left.val and root.val > root.right.val

# Bài 4
def count_nodes(root):
    if root is None:
        return 0
    return 1 + count_nodes(root.left) + count_nodes(root.right)

# Bài 5
def is_sum_tree(root):
    if root is None or (root.left is None and root.right is None):
        return True
    total = (root.left.val if root.left else 0) + (root.right.val if root.right else 0)
    return root.val == total and is_sum_tree(root.left) and is_sum_tree(root.right)

assert root_greater_than_children(build_tree([10, 4, 6])) is True
assert root_greater_than_children(build_tree([5, 7, 1])) is False
assert count_nodes(build_tree([10, 4, 6])) == 3
assert count_nodes(build_tree([1, 2, 3, 4, 5])) == 5
assert count_nodes(None) == 0
assert is_sum_tree(build_tree([10, 4, 6, 1, 3])) is True
assert is_sum_tree(build_tree([10, 4, 6, 1, 2])) is False
assert is_sum_tree(build_tree([9])) is True
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cây 3 nút: so sánh trực tiếp ba giá trị)

```python
class Solution:
    def checkTree(self, root: Optional[TreeNode]) -> bool:
        return root.val == root.left.val + root.right.val
```

**Checklist**

- [ ] Có dùng `.val` cho cả ba nút (không cộng đối tượng nút)?
- [ ] Dùng `==` (so sánh) chứ không phải `=` (gán)?
- [ ] Đã đọc constraints: đủ ba nút nên `left`, `right` không `None`?
- [ ] Nếu làm biến thể cây thiếu con: đã kiểm tra `None` hoặc dùng hàm `value`?
- [ ] Đã thử số âm, số 0, biên `±100`, và phản ví dụ `[100,100,100]`?
- [ ] Signature đúng `checkTree(self, root: Optional[TreeNode]) -> bool` và có `from typing import Optional`?
- [ ] Biết rằng trên LeetCode `TreeNode` đã có sẵn, chỉ khi chạy local mới cần tự định nghĩa?